# 2.7 Warnings: pretrained Roboflow mould models on our own test photos

**What this notebook does.** It sends the same test photos to three existing Roboflow models and compares them at **image level**: does the model say "mould" or not? It is the mould part of **Warnings**. **Inference only: nothing is trained.**

| Key | Roboflow model ID | Universe page |
|---|---|---|
| mycoguard | `identifying-contamination-in-myc/1` | taos-workspace-7tzlc / identifying-contamination-in-myc |
| fungi_buster | `disease-mushroom/1` | fungi-buster-1hvwz / disease-mushroom |
| mushroom_conditions | `mushroom-conditions-dataset/1` | abdullah-hzljd / mushroom-conditions-dataset |

**Read this before trusting any number.**
- **The truth is provisional.** `clean/` = no mould and `mould/` = mould (folder names, not lab results). The `cobweb/` folder is deliberately excluded from this comparison; only `clean/` and non-cobweb `mould/` photos are scored.
- Image level only. No mAP, no IoU, no box recall: there are no reviewed boxes.
- An image counts as **contaminated** when at least one box mapped to `suspected_contamination` has a confidence at or above the threshold. A `healthy` box never cancels it. Classes you have not mapped are **never** counted as mould.
- Thresholds 0.25 (loose) and **0.50 (primary)**.
- A failed call is left out of the counts (not treated as "no mould").
- Photos flagged `d005`, `syldd` or `possible_roboflow_export` may overlap with training sets or with our own live batch: the comparison also shows the numbers without them.
- The test set is small. Differences of a photo or two are noise: **no ranking**.
- `mushroom-conditions-dataset/1` may be a whole-image classifier, not a box detector. It is reported by the same image rule but is not like-for-like with the detectors.

**Later.** Our own model is added as one more row (same cache layout). The same `clean/` and `mould/` folders are used for every model. `cobweb/` is recorded as excluded and is never sent to a model in this run; calls are cached per photo hash.

**How to run (Kaggle).**
1. Add the dataset `mould-test-photos` (Add Input); it must contain `clean/`, `mould/`, and `cobweb/`. This run scores `clean/` and `mould/` only and skips `cobweb/`. Add-ons, Secrets: switch on `ROBOFLOW_API_KEY`.
2. Run the cells down to the **Preflight** cell. It prints the image count x model count and which models answer. It sends one probe photo per model.
3. Only then run the **Inference** cell (set `CONFIRM_RUN = True` first).
4. Set the class map (cell "Class map"), then run the analysis cells. Download `/kaggle/working/mould_benchmark`.

The key is read from the secret and sent in a request header. It is never printed, saved, logged or put in a URL.

In [ ]:
# 1. Settings: change these lines only if you want to
import os, re, json, time, hashlib, math, datetime
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path(os.environ.get("MOULD_OUTPUT_DIR") or ("/kaggle/working/mould_benchmark" if ON_KAGGLE else "mould_benchmark"))  # same cache path reuses yesterday's replies
IMAGE_ROOT = os.environ.get("MOULD_IMAGE_ROOT") or None   # None = find clean/, mould/ and cobweb/ under /kaggle/input
LABEL_DIRS = {"clean": "healthy", "mould": "contaminated"}   # only these folders are scored
EXCLUDED_LABEL_FOLDERS = ("cobweb",)  # folder is required for this dataset check, but its photos are not scored
REQUIRED_FOLDERS = tuple(LABEL_DIRS) + EXCLUDED_LABEL_FOLDERS
LABEL_NOTE = "provisional folder labels (clean/ = no mould, mould/ = mould); cobweb/ excluded; not lab-confirmed"

THRESHOLDS = (0.25, 0.50)
PRIMARY = 0.50
CONF_FLOOR = 0.05            # lowest confidence asked from the server; thresholds are applied afterwards, so one call serves both
SECRET_NAME = "ROBOFLOW_API_KEY"
API_BASE = "https://serverless.roboflow.com"
TIMEOUT_S, RETRIES, BACKOFF_S = 60, 3, 2.0
GIVE_UP_AFTER = 5            # this many failures in a row with the same non-retryable error: model marked unavailable

# Models (verify each: the preflight reports which ones really answer; nothing is substituted)
MODELS = [
    dict(key="mycoguard", name="MycoGuard (identifying-contamination-in-myc/1)", model_id="identifying-contamination-in-myc/1",
         workspace="taos-workspace-7tzlc", universe="https://universe.roboflow.com/taos-workspace-7tzlc/identifying-contamination-in-myc"),
    dict(key="fungi_buster", name="Fungi Buster (disease-mushroom/1)", model_id="disease-mushroom/1",
         workspace="fungi-buster-1hvwz", universe="https://universe.roboflow.com/fungi-buster-1hvwz/disease-mushroom"),
    dict(key="mushroom_conditions", name="Mushroom Conditions (mushroom-conditions-dataset/1)", model_id="mushroom-conditions-dataset/1",
         workspace="abdullah-hzljd", universe="https://universe.roboflow.com/abdullah-hzljd/mushroom-conditions-dataset"),
]

# Class map: their class name -> suspected_contamination / healthy / ignore.  Case does not matter.
# Leave a model's map empty until the class table (cell "Class map") has shown its real class names.
# Example:  "mycoguard": {"Trichoderma": "suspected_contamination", "Healthy Mycelium": "healthy"},
CLASS_MAP = {
    "mycoguard": {},
    "fungi_buster": {},
    "mushroom_conditions": {},
}
ALLOWED = {"suspected_contamination", "healthy", "ignore"}

# Photo flags (file-name rules; a flagged photo may overlap with a training set or our own batch)
FLAG_RULES = {
    "d005": r"\d{6}_[A-Za-z]+_Tv",                 # d005 dishes (possible source overlap; not used for training here)
    "syldd": r"^IMG_2026\d{4}_\d{6}",              # our live batch (phone photos)
    "possible_roboflow_export": r"\.rf\.|_aug_",   # augmented-export names
    "baibua": r"baibua",
}
SOURCE_HINTS = ("mendeley_", "reddit_", "openverse_", "shroomok_")
HASH_DISTANCE = 4            # near-duplicate: 64-bit difference hashes that differ in <= this many bits

CACHE_DIR = OUTPUT_DIR / "cache"
for _d in (OUTPUT_DIR, OUTPUT_DIR / "figures", CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ERR_LOG = CACHE_DIR / "_errors.jsonl"
try:
    display
except NameError:
    display = print
print("output:", OUTPUT_DIR)

In [ ]:
# 2. Helpers (no network here)
import numpy as np, pandas as pd
from collections import Counter
from PIL import Image, ImageOps

class StopRun(Exception):
    pass

def stop(msg):
    print("STOP:", msg)
    raise StopRun(msg)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

def load_rgb(p):
    # The same pixels go to every model and into the figures: EXIF turn applied, RGB.
    with Image.open(p) as im:
        return ImageOps.exif_transpose(im).convert("RGB")

def dhash(im, n=8):
    g = np.asarray(im.convert("L").resize((n + 1, n), Image.LANCZOS), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

def find_root():
    if IMAGE_ROOT:
        return Path(IMAGE_ROOT)
    for base in ("/kaggle/input", "../data/raw/test_image_contamination", "data/raw/test_image_contamination"):
        if not os.path.isdir(base):
            continue
        for dp, dn, _ in os.walk(base):
            if all(k in dn for k in REQUIRED_FOLDERS):
                return Path(dp)
            if dp[len(base):].count(os.sep) >= 4:
                dn[:] = []
    return None

def file_flags(name):
    return [k for k, rx in FLAG_RULES.items() if re.search(rx, name, re.I)]

def build_manifest(root):
    rows, skipped = [], []
    for folder, label in LABEL_DIRS.items():
        for p in sorted((root / folder).iterdir()):
            if not p.is_file() or p.name.startswith(".") or ":Zone.Identifier" in p.name:
                continue
            try:
                with Image.open(p) as im:
                    fmt, (w, h) = im.format, im.size
                im = load_rgb(p)
            except Exception as e:
                skipped.append((f"{folder}/{p.name}", f"{type(e).__name__}"))
                continue
            rows.append(dict(image_id=f"{folder}/{p.name}", folder=folder, label=label, label_basis=LABEL_NOTE,
                             sha256=sha256_file(p), extension=p.suffix.lower() or "(none)", real_format=fmt,
                             width=w, height=h, bytes=p.stat().st_size,
                             photo_flags=";".join(file_flags(p.name)),
                             source_hint=next((s.rstrip("_") for s in SOURCE_HINTS if p.name.startswith(s)), ""),
                             dhash_bits=dhash(im), local_path=str(p)))
    m = pd.DataFrame(rows)
    if m.empty:
        return m, skipped
    first = {}
    m["duplicate_of"] = ""
    for i, r in m.iterrows():
        if r.sha256 in first:
            m.at[i, "duplicate_of"] = first[r.sha256]
        else:
            first[r.sha256] = r.image_id
    m["near_duplicate_of"] = ""
    keep = m[m.duplicate_of == ""]
    for a in range(len(keep)):
        for b in range(a):
            ra, rb = keep.iloc[a], keep.iloc[b]
            if (ra.dhash_bits != rb.dhash_bits).sum() <= HASH_DISTANCE and m.at[ra.name, "near_duplicate_of"] == "":
                m.at[ra.name, "near_duplicate_of"] = rb.image_id
    m["scored"] = m.duplicate_of == ""        # exact copies are scored once
    return m, skipped

# ---- key (never printed, never saved) ----
def get_key():
    key = None
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret(SECRET_NAME)
    except Exception:
        pass
    return key or os.environ.get(SECRET_NAME)

def scrub(text, key):
    text = str(text)
    return text.replace(key, "***") if key else text

# ---- one model call, cached per (model ID, image SHA256) ----
def encode_jpeg(path):
    import io
    im = load_rgb(path)
    for q, scale in ((95, 1.0), (90, 0.75), (85, 0.5)):
        im2 = im if scale == 1.0 else im.resize((int(im.width * scale), int(im.height * scale)))
        buf = io.BytesIO(); im2.save(buf, "JPEG", quality=q)
        if buf.tell() < 19_000_000:
            return buf.getvalue(), im.size
    return buf.getvalue(), im.size

def cache_path(model, sha):
    return CACHE_DIR / model["model_id"].replace("/", "__") / f"{sha}.json"

def read_cache(model, sha):
    p = cache_path(model, sha)
    if not p.exists():
        return None
    try:
        d = json.loads(p.read_text())
    except Exception:
        return None
    return d if d.get("confidence_floor", 1) <= CONF_FLOOR + 1e-9 else None

def log_error(model, row, status, error, attempts):
    with open(ERR_LOG, "a") as f:
        f.write(json.dumps(dict(when=datetime.datetime.now().isoformat(timespec="seconds"), model_key=model["key"],
                                model_id=model["model_id"], image_id=row.image_id, sha256=row.sha256,
                                http_status=status, error=error, attempts=attempts)) + "\n")

def call_model(model, jpg, key):
    import requests
    url = f"{API_BASE}/{model['model_id']}"        # no key in the URL
    err, status = "", None
    for attempt in range(1, RETRIES + 1):
        retry = True
        try:
            r = requests.post(url, params={"confidence": CONF_FLOOR}, headers={"Authorization": f"Bearer {key}"},
                              files={"file": ("image.jpg", jpg, "image/jpeg")}, timeout=TIMEOUT_S)
            status = r.status_code
            if status == 200:
                try:
                    js = r.json()
                except ValueError:
                    js = None
                if isinstance(js, dict) and "predictions" in js:
                    return dict(ok=True, status=status, response=js, error="", attempts=attempt)
                err, retry = f"HTTP 200 but no 'predictions' in the reply: {scrub(r.text, key)[:300]}", False
            else:
                err = f"HTTP {status}: {scrub(r.text, key)[:300]}"
                retry = status in (408, 429, 500, 502, 503, 504)
        except requests.RequestException as e:
            err = f"{type(e).__name__}: {scrub(e, key)[:300]}"
        if not retry:
            break
        time.sleep(BACKOFF_S * attempt)
    return dict(ok=False, status=status, response=None, error=err, attempts=attempt)

def run_one(model, row, key):
    d = read_cache(model, row.sha256)
    if d:
        return dict(ok=True, cached=True, status=200, error="")
    jpg, size = encode_jpeg(row.local_path)
    res = call_model(model, jpg, key)
    if res["ok"]:
        p = cache_path(model, row.sha256)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(dict(model_id=model["model_id"], sha256=row.sha256, image_id=row.image_id, confidence_floor=CONF_FLOOR,
                                     fetched_at=datetime.datetime.now().isoformat(timespec="seconds"),
                                     sent_size=list(size), response=res["response"])))
    else:
        log_error(model, row, res["status"], res["error"], res["attempts"])
    return dict(ok=res["ok"], cached=False, status=res["status"], error=res["error"])

# ---- reading the replies ----
def parse_predictions(resp):
    out, kind = [], "unknown"
    preds = resp.get("predictions")
    if isinstance(preds, list):
        for p in preds:
            if not isinstance(p, dict):
                continue
            if "x" in p and "width" in p:
                kind = "detector"
                out.append(dict(cls=str(p.get("class", "")), conf=float(p.get("confidence", 0)), x=p["x"], y=p["y"], w=p["width"], h=p["height"]))
            elif "class" in p:
                kind = "classifier"
                out.append(dict(cls=str(p["class"]), conf=float(p.get("confidence", 0)), x=None, y=None, w=None, h=None))
    elif isinstance(preds, dict):                      # multi-label classifier
        kind = "classifier"
        for c, v in preds.items():
            out.append(dict(cls=str(c), conf=float(v.get("confidence", 0)), x=None, y=None, w=None, h=None))
    if kind == "unknown":
        kind = "classifier" if ("top" in resp or "predicted_classes" in resp) else "detector"
    return out, kind

def map_class(model_key, cls):
    m = {k.strip().lower(): v for k, v in CLASS_MAP.get(model_key, {}).items()}
    for k, v in m.items():
        if v not in ALLOWED:
            stop(f"CLASS_MAP[{model_key!r}][{k!r}] = {v!r}: use one of {sorted(ALLOWED)}")
    return m.get(cls.strip().lower())          # None = not mapped yet: never counted as mould

SUGGEST_CONTAM = ("mold", "mould", "trichoderma", "cobweb", "contamin", "bacteri", "blotch", "disease", "rot", "mildew",
                  "penicillium", "aspergillus", "neurospora", "fusarium", "infect")
SUGGEST_HEALTHY = ("healthy", "normal", "fresh", "good", "clean", "mycelium")
def suggest(cls):
    c = cls.lower()
    if any(k in c for k in SUGGEST_HEALTHY) and not any(k in c for k in ("unhealthy", "not ")):
        return "healthy"
    return "suspected_contamination" if any(k in c for k in SUGGEST_CONTAM) else "?"

# ---- metrics ----
def ratio(n, d):
    return n / d if d else float("nan")

def wilson(k, n, z=1.96):
    if not n:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))

def confusion_metrics(tp, fp, tn, fn):
    P, N = tp + fn, tn + fp
    sens, spec, prec = ratio(tp, P), ratio(tn, N), ratio(tp, tp + fp)
    bal = (sens + spec) / 2 if P and N else float("nan")
    return dict(tp=tp, fp=fp, tn=tn, fn=fn, n_positive=P, n_negative=N,
                sensitivity=sens, sensitivity_frac=f"{tp}/{P}",
                specificity=spec, specificity_frac=f"{tn}/{N}",
                precision=prec, precision_frac=f"{tp}/{tp + fp}",
                balanced_accuracy=bal, balanced_accuracy_frac=f"({tp}/{P} + {tn}/{N})/2")

def pct(x, frac):
    return f"NaN ({frac})" if x != x else f"{x * 100:.0f}% ({frac})"
print("helpers ready")

## Metric check on a small hand-made example
No network. Ten photos: 5 contaminated, 5 healthy. The model finds 3 of the 5 contaminated and wrongly flags 1 of the 5 healthy: TP 3, FN 2, FP 1, TN 4. Expected: sensitivity 3/5 = 60%, specificity 4/5 = 80%, precision 3/4 = 75%, balanced accuracy 70%. A case with no healthy photos must give NaN, not 0.

In [ ]:
# 3. Check the metric code on the hand-made example
labels = ["contaminated"] * 5 + ["healthy"] * 5
flagged = [True, True, True, False, False, True, False, False, False, False]
tp = sum(l == "contaminated" and f for l, f in zip(labels, flagged)); fn = sum(l == "contaminated" and not f for l, f in zip(labels, flagged))
fp = sum(l == "healthy" and f for l, f in zip(labels, flagged)); tn = sum(l == "healthy" and not f for l, f in zip(labels, flagged))
m = confusion_metrics(tp, fp, tn, fn)
assert (tp, fp, tn, fn) == (3, 1, 4, 2), (tp, fp, tn, fn)
assert abs(m["sensitivity"] - 0.6) < 1e-12 and m["sensitivity_frac"] == "3/5"
assert abs(m["specificity"] - 0.8) < 1e-12 and m["specificity_frac"] == "4/5"
assert abs(m["precision"] - 0.75) < 1e-12 and m["precision_frac"] == "3/4"
assert abs(m["balanced_accuracy"] - 0.7) < 1e-12
z = confusion_metrics(0, 0, 0, 0)                     # nothing scored: every ratio is NaN
assert all(math.isnan(z[k]) for k in ("sensitivity", "specificity", "precision", "balanced_accuracy"))
z2 = confusion_metrics(0, 0, 4, 0)                    # no contaminated photos: sensitivity NaN, precision NaN
assert math.isnan(z2["sensitivity"]) and math.isnan(z2["precision"]) and z2["specificity"] == 1.0 and math.isnan(z2["balanced_accuracy"])
print("metric check passed:", pct(m["sensitivity"], m["sensitivity_frac"]), pct(m["specificity"], m["specificity_frac"]),
      pct(m["precision"], m["precision_frac"]), pct(m["balanced_accuracy"], m["balanced_accuracy_frac"]))

## Step 1. The photos
Finds the uploaded dataset, scores photos from `clean/` and `mould/`, and excludes every photo in `cobweb/`. It hashes the scored photos, flags exact and near copies and possible source overlap, then writes `image_manifest.csv` plus an `excluded_photos.csv` audit. It stops if the expected three folders are missing.

In [ ]:
# 4. Image manifest (no key, no network)
ROOT = find_root()
if ROOT is None or not all((ROOT / d).is_dir() for d in REQUIRED_FOLDERS):
    stop("test photos not found. Add the Kaggle input 'mould-test-photos' with clean/, mould/, and cobweb/ folders. "
         "Only clean/ and mould/ are scored; cobweb/ is excluded. Elsewhere, set MOULD_IMAGE_ROOT to that dataset folder.")
MANIFEST, SKIPPED = build_manifest(ROOT)
# Record the excluded cobweb folder so the output proves which photos were deliberately left out.
excluded_rows = []
for folder in EXCLUDED_LABEL_FOLDERS:
    for p in sorted((ROOT / folder).iterdir()):
        if not p.is_file() or p.name.startswith(".") or ":Zone.Identifier" in p.name:
            continue
        row = dict(image_id=f"{folder}/{p.name}", folder=folder, excluded_reason="folder_excluded_from_this_comparison",
                   sha256=sha256_file(p), readable_image=False)
        try:
            with Image.open(p) as im:
                row.update(readable_image=True, real_format=im.format, width=im.width, height=im.height)
        except Exception as e:
            row["read_error"] = type(e).__name__
        excluded_rows.append(row)
EXCLUDED = pd.DataFrame(excluded_rows, columns=["image_id", "folder", "excluded_reason", "sha256", "readable_image", "real_format", "width", "height", "read_error"])
EXCLUDED.to_csv(OUTPUT_DIR / "excluded_photos.csv", index=False)
if MANIFEST.empty:
    stop(f"no readable photos under {ROOT}")
MANIFEST.drop(columns=["dhash_bits", "local_path"]).to_csv(OUTPUT_DIR / "image_manifest.csv", index=False)
SCORED = MANIFEST[MANIFEST.scored].reset_index(drop=True)
print("photo folder:", ROOT)
print(f"Scoring clean/ + mould/: {len(MANIFEST)} readable files before duplicate removal; excluded cobweb/: {int(EXCLUDED.readable_image.sum())} readable image(s) across {len(EXCLUDED)} file(s).")
display(MANIFEST.groupby(["folder", "label"]).agg(files=("image_id", "size"), scored=("scored", "sum"),
        flagged=("photo_flags", lambda s: int((s != "").sum()))).reset_index())
display(MANIFEST.groupby(["folder", "extension"]).size().rename("files").reset_index())
if SKIPPED:
    print("NOT READABLE as images (left out):", SKIPPED)
for col, text in (("duplicate_of", "exact copy (same SHA256, scored once)"), ("near_duplicate_of", "near copy (similar picture, still scored: check by eye)")):
    d = MANIFEST[MANIFEST[col] != ""]
    if len(d):
        print(text); display(d[["image_id", col]])
fl = MANIFEST[MANIFEST.photo_flags != ""]
if len(fl):
    print("flagged photos (possible overlap with training sets or our own batch):"); display(fl[["image_id", "photo_flags"]])
print(f"\nLabels are {LABEL_NOTE}.")

## Step 2. Preflight (read this output before running inference)
Prints **images x models = calls**, whether the key is present (yes/no only), and which models answer. To know whether a model answers it sends **one probe photo per model** (the first scored photo; cached, so it is not paid for twice). A model that fails is marked unavailable with the real error and the others carry on.

In [ ]:
# 5. Preflight
KEY = get_key()
if not KEY:
    stop(f"no Roboflow key. Kaggle: Add-ons > Secrets, add {SECRET_NAME} and switch it on for this notebook. Elsewhere: set the env var {SECRET_NAME}.")
print(f"key {SECRET_NAME}: found (value not shown)")
if "SCORED" not in globals():
    stop("run the Image manifest cell first")
n_img, n_mod = len(SCORED), len(MODELS)
print(f"images to score: {n_img} ({(SCORED.label == 'contaminated').sum()} contaminated-folder, {(SCORED.label == 'healthy').sum()} healthy-folder)")
print(f"models: {n_mod}\ncalls if every model answers: {n_img} x {n_mod} = {n_img * n_mod} (fewer when cached)")
PROBE = SCORED.iloc[0]
rows = []
for m in MODELS:
    r = run_one(m, PROBE, KEY)
    kind, classes = "", ""
    if r["ok"]:
        preds, kind = parse_predictions(read_cache(m, PROBE.sha256)["response"])
        classes = ", ".join(sorted({p["cls"] for p in preds})) or "(no predictions on the probe photo)"
    rows.append(dict(model=m["key"], model_id=m["model_id"], answers="YES" if r["ok"] else "NO", kind=kind,
                     probe_classes=classes, real_error=r["error"]))
PREFLIGHT = pd.DataFrame(rows)
display(PREFLIGHT)
print("probe photo:", PROBE.image_id)
if not (PREFLIGHT.answers == "YES").any():
    print("No model answered. Read 'real_error' above; fix the model ID or the key before going on.")
else:
    print("Next: set CONFIRM_RUN = True in the Inference cell and run it on purpose.")

## Step 3. Inference (run on purpose)
Calls each available model once per photo (up to 3 tries per call) and caches the raw reply per (model ID, photo SHA256), without the key. Cached photos are skipped, so a second run costs nothing. **This is the only cell that spends Roboflow calls.**

In [ ]:
# 6. Inference: set CONFIRM_RUN = True to run
CONFIRM_RUN = False

if not CONFIRM_RUN:
    print("Not run. Check the preflight output, then set CONFIRM_RUN = True and run this cell again.")
else:
    if "PREFLIGHT" not in globals():
        stop("run the Preflight cell first")
    answering = set(PREFLIGHT[PREFLIGHT.answers == "YES"].model)
    summary = []
    for m in MODELS:
        if m["key"] not in answering:
            summary.append((m["key"], "skipped: did not answer in the preflight", 0, 0, 0))
            continue
        ok = fail = cached = streak = 0
        last_err = ""
        for i, row in enumerate(SCORED.itertuples(index=False), 1):
            r = run_one(m, row, KEY)
            if r["ok"]:
                ok += 1; cached += r["cached"]; streak = 0
            else:
                fail += 1
                streak = streak + 1 if (r["error"] == last_err or streak == 0) else 1
                last_err = r["error"]
                if r["status"] not in (None, 408, 429, 500, 502, 503, 504) and streak >= GIVE_UP_AFTER:
                    print(f"{m['key']}: {streak} identical errors in a row, giving up on this model: {r['error']}")
                    break
            if i % 10 == 0 or i == len(SCORED):
                print(f"{m['key']}: {i}/{len(SCORED)}  ok {ok}  failed {fail}")
        summary.append((m["key"], "done", ok, cached, fail))
    display(pd.DataFrame(summary, columns=["model", "state", "answered", "of_which_cached", "failed"]))

## Step 4. Class map
Their class names differ per model. The table shows every class each model actually returned, how often, what the map says now, and a keyword **suggestion** (a hint, not applied). Copy your decisions into `CLASS_MAP` in the settings cell, then run this cell and the analysis again: nothing is called twice. Unmapped classes are never counted as mould, and a model with no class mapped to `suspected_contamination` is left out of the metrics with a message.

In [ ]:
# 7. Class map table (reads the cache only)
def load_cache_table():
    if "SCORED" not in globals():
        stop("run the Image manifest cell first")
    rec = []
    for m in MODELS:
        for r in SCORED.itertuples(index=False):
            d = read_cache(m, r.sha256)
            if d is None:
                continue
            preds, kind = parse_predictions(d["response"])
            rec.append(dict(model_key=m["key"], image_id=r.image_id, sha256=r.sha256, kind=kind, preds=preds, size=d.get("sent_size")))
    return rec
CACHED = load_cache_table()
cls_rows = Counter((c["model_key"], p["cls"]) for c in CACHED for p in c["preds"])
if not cls_rows:
    print("No cached replies yet: run the Preflight and Inference cells first.")
else:
    tab = pd.DataFrame([dict(model=k, their_class=c, n_predictions=n, mapped_to=map_class(k, c) or "NOT MAPPED (never counted as mould)",
                             suggestion=suggest(c)) for (k, c), n in sorted(cls_rows.items())])
    display(tab)
    print("Copy into CLASS_MAP, e.g.: CLASS_MAP['%s'] = {...}" % tab.model.iloc[0])
    print("Check by eye: the qualitative grid (below) shows what each class looks like on the photos.")

## Step 5. Results and exports
Builds the tables, figures and summary from the **cached replies only** (no calls); results include `clean/` and non-cobweb `mould/` photos only. Rules again: contaminated = at least one mapped mould box at or above the threshold; `healthy` never overrides; failed photos are left out of the counts; numbers are on the photos **every usable model answered**, and the gaps are listed.

In [ ]:
# 8. Image-level results, metrics, registry and error list
def read_errors():
    if not ERR_LOG.exists():
        return pd.DataFrame(columns=["when", "model_key", "model_id", "image_id", "sha256", "http_status", "error", "attempts"])
    return pd.DataFrame([json.loads(l) for l in ERR_LOG.read_text().splitlines() if l.strip()])

ERRS = read_errors()
today = datetime.date.today().isoformat()
CACHED = load_cache_table()
by_model = {m["key"]: {c["sha256"]: c for c in CACHED if c["model_key"] == m["key"]} for m in MODELS}

# ---- registry ----
reg, usable = [], []
for m in MODELS:
    n_ok = len(by_model[m["key"]])
    e = ERRS[ERRS.model_key == m["key"]]
    kinds = Counter(c["kind"] for c in by_model[m["key"]].values())
    classes = sorted({p["cls"] for c in by_model[m["key"]].values() for p in c["preds"]})
    mapped_mould = [c for c in classes if map_class(m["key"], c) == "suspected_contamination"]
    if n_ok == 0:
        status = "unavailable" if len(e) else "not run yet"
    elif not mapped_mould:
        status = "answered; class map not set (left out of the metrics)"
    else:
        status = "usable"; usable.append(m["key"])
    reg.append(dict(model_key=m["key"], name=m["name"], model_id=m["model_id"], workspace=m["workspace"], universe_page=m["universe"],
                    call_url=f"{API_BASE}/{m['model_id']}", model_kind=(kinds.most_common(1)[0][0] if kinds else ""),
                    status=status, images_answered=n_ok, images_failed=int(e[~e.sha256.isin(by_model[m['key']])].sha256.nunique()) if len(e) else 0,
                    last_real_error=(e.iloc[-1].error if len(e) and n_ok == 0 else ""), classes_seen="; ".join(classes),
                    classes_mapped_to_mould="; ".join(mapped_mould), architecture="not stated in the reply (not verified)",
                    confidence_floor=CONF_FLOOR, checked_on=today))
REG = pd.DataFrame(reg)
REG.to_csv(OUTPUT_DIR / "model_registry.csv", index=False)
display(REG[["model_key", "model_id", "model_kind", "status", "images_answered", "images_failed", "last_real_error"]])

HAVE_RESULTS = bool(usable)
if not HAVE_RESULTS:
    print("No usable model yet (no replies, or no class mapped to suspected_contamination): tables and figures are not built.")
else:
    pred_rows, img_rows = [], []
    for m in MODELS:
        for r in SCORED.itertuples(index=False):
            c = by_model[m["key"]].get(r.sha256)
            base = dict(model_key=m["key"], image_id=r.image_id, label=r.label, photo_flags=r.photo_flags)
            if c is None:
                e = ERRS[(ERRS.model_key == m["key"]) & (ERRS.sha256 == r.sha256)]
                st = "model_unavailable" if REG.set_index("model_key").status[m["key"]] == "unavailable" else ("failed" if len(e) else "not_run")
                img_rows.append(dict(base, status=st, error=(e.iloc[-1].error if len(e) else "")))
                continue
            for p in c["preds"]:
                pred_rows.append(dict(model_key=m["key"], image_id=r.image_id, sha256=r.sha256, their_class=p["cls"], confidence=p["conf"],
                                      mapped_to=map_class(m["key"], p["cls"]) or "not_mapped", x=p["x"], y=p["y"], width=p["w"], height=p["h"]))
            row = dict(base, status="ok", error="")
            mould = [p for p in c["preds"] if map_class(m["key"], p["cls"]) == "suspected_contamination"]
            heal = [p for p in c["preds"] if map_class(m["key"], p["cls"]) == "healthy"]
            row["max_mould_conf"] = max((p["conf"] for p in mould), default=0.0)
            row["max_healthy_conf"] = max((p["conf"] for p in heal), default=0.0)
            row["unmapped_classes"] = ";".join(sorted({p["cls"] for p in c["preds"] if map_class(m["key"], p["cls"]) is None}))
            for th in THRESHOLDS:
                pos = row["max_mould_conf"] >= th            # a healthy box never overrides a mould box
                row[f"pred_contaminated_{th:.2f}"] = pos
                row[f"outcome_{th:.2f}"] = ("TP" if pos else "FN") if r.label == "contaminated" else ("FP" if pos else "TN")
            img_rows.append(row)
    PRED = pd.DataFrame(pred_rows, columns=["model_key", "image_id", "sha256", "their_class", "confidence", "mapped_to", "x", "y", "width", "height"])
    IMG = pd.DataFrame(img_rows)
    PRED.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    IMG.to_csv(OUTPUT_DIR / "image_level_results.csv", index=False)

    # ---- same photos for every usable model ----
    ok_sets = [set(IMG[(IMG.model_key == k) & (IMG.status == "ok")].image_id) for k in usable]
    COMMON = set.intersection(*ok_sets)
    flag_of = SCORED.set_index("image_id").photo_flags.to_dict()
    gaps = {k: sorted(set(SCORED.image_id) - set(IMG[(IMG.model_key == k) & (IMG.status == "ok")].image_id)) for k in usable}
    print(f"photos answered by every usable model: {len(COMMON)} of {len(SCORED)}")
    for k, g in gaps.items():
        if g:
            print(f"  gap, {k} missing {len(g)} photo(s): {g[:6]}{' ...' if len(g) > 6 else ''}")

    comp = []
    for m in MODELS:
        k = m["key"]
        if k not in usable:
            continue
        sub = IMG[IMG.model_key == k]
        for subset, keep in (("all_common", lambda i: True), ("without_flagged", lambda i: flag_of[i] == "")):
            ids = {i for i in COMMON if keep(i)}
            s = sub[sub.image_id.isin(ids)]
            for th in THRESHOLDS:
                o = s[f"outcome_{th:.2f}"].value_counts()
                cm = confusion_metrics(int(o.get("TP", 0)), int(o.get("FP", 0)), int(o.get("TN", 0)), int(o.get("FN", 0)))
                lo_s, hi_s = wilson(cm["tp"], cm["n_positive"]); lo_p, hi_p = wilson(cm["tn"], cm["n_negative"])
                comp.append(dict(model_key=k, name=m["name"], model_kind=REG.set_index("model_key").model_kind[k], subset=subset,
                                 threshold=th, is_primary=(th == PRIMARY), n_images=len(ids), **cm,
                                 sensitivity_ci95=f"{lo_s:.2f}-{hi_s:.2f}", specificity_ci95=f"{lo_p:.2f}-{hi_p:.2f}",
                                 images_missing_vs_all_scored=len(gaps[k]),
                                 note="small set: no ranking" if min(cm["n_positive"], cm["n_negative"]) < 30 else ""))
    COMP = pd.DataFrame(comp)
    COMP.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

    # ---- errors and failure cases ----
    ef = []
    for e in ERRS.itertuples(index=False):
        ef.append(dict(kind="api_error", model_key=e.model_key, image_id=e.image_id, detail=f"HTTP {e.http_status}: {e.error}"))
    for _, r in IMG[(IMG.status == "ok")].iterrows():
        for th in THRESHOLDS:
            oc = r[f"outcome_{th:.2f}"]
            if oc in ("FP", "FN"):
                ef.append(dict(kind=f"{oc}_at_{th:.2f}", model_key=r.model_key, image_id=r.image_id,
                               detail=f"folder label {r.label}; max mould conf {r.max_mould_conf:.2f}; flags {r.photo_flags or '-'}"))
    for r in IMG[(IMG.status == "ok") & (IMG.unmapped_classes != "")].itertuples(index=False):
        ef.append(dict(kind="unmapped_class", model_key=r.model_key, image_id=r.image_id, detail=r.unmapped_classes))
    for k, g in gaps.items():
        for i in g:
            ef.append(dict(kind="not_scored_by_model", model_key=k, image_id=i, detail="left out of the counts, not treated as no mould"))
    EF = pd.DataFrame(ef, columns=["kind", "model_key", "image_id", "detail"])
    EF.to_csv(OUTPUT_DIR / "errors_and_failure_cases.csv", index=False)

    show = COMP[(COMP.subset == "all_common")][["model_key", "model_kind", "threshold", "tp", "fp", "tn", "fn",
            "sensitivity_frac", "sensitivity", "specificity_frac", "specificity", "precision_frac", "precision", "balanced_accuracy"]]
    display(show.round(2))
    print(f"Labels are {LABEL_NOTE}. Small set: do not rank models on differences of a photo or two.")

## Step 6. Figures
Slide-sized (13.33 x 7.5 in). Counts and denominators are written on the bars.

In [ ]:
# 9. Figures: model_comparison.png, confusion_matrices.png
import matplotlib
import matplotlib.pyplot as plt
FIG = OUTPUT_DIR / "figures"
if not HAVE_RESULTS:
    print("No results yet: figures not built.")
else:
    plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
    base = COMP[COMP.subset == "all_common"]
    keys = list(dict.fromkeys(base.model_key))
    n_pos, n_neg = int(base.n_positive.iloc[0]), int(base.n_negative.iloc[0])
    metrics = [("sensitivity", "Mould found (sensitivity)", "#c0504d"), ("specificity", "Healthy kept clear (specificity)", "#4f81bd"),
               ("balanced_accuracy", "Balanced accuracy", "#7f7f7f")]
    fig, axes = plt.subplots(1, 2, figsize=(13.33, 7.5), sharey=True)
    for ax, th in zip(axes, sorted(THRESHOLDS, reverse=True)):
        s = base[base.threshold == th].set_index("model_key")
        w = 0.26
        for j, (col, lab, colr) in enumerate(metrics):
            for i, k in enumerate(keys):
                v = s.loc[k, col]; frac = s.loc[k, col + "_frac"]
                v0 = 0 if v != v else v
                ax.bar(i + (j - 1) * w, v0, w * 0.92, color=colr, label=lab if i == 0 else None)
                ax.text(i + (j - 1) * w, v0 + 0.02, ("NaN" if v != v else f"{v * 100:.0f}%") + f"\n{frac}", ha="center", va="bottom", fontsize=7)
        ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys)
        ax.set_ylim(0, 1.25); ax.set_title(f"confidence >= {th:.2f}" + ("  (primary)" if th == PRIMARY else ""))
        ax.axhline(1, color="#cccccc", lw=0.8)
    axes[0].set_ylabel("share of photos"); axes[0].legend(loc="upper left", fontsize=9, frameon=False)
    fig.suptitle(f"Pretrained Roboflow models on our test photos without cobweb: {n_pos} mould-folder and {n_neg} clean-folder photos answered by every model", fontsize=14)
    fig.text(0.5, 0.01, f"Labels are {LABEL_NOTE}. Image level only. Small set: differences of a photo or two are not a ranking.", ha="center", fontsize=10)
    fig.tight_layout(rect=(0, 0.04, 1, 0.95)); fig.savefig(FIG / "model_comparison.png", dpi=150); plt.show()

    s = base[base.threshold == PRIMARY].set_index("model_key")
    fig, axes = plt.subplots(1, len(keys), figsize=(13.33, 7.5), squeeze=False)
    for ax, k in zip(axes[0], keys):
        r = s.loc[k]
        mat = np.array([[r.tp, r.fn], [r.fp, r.tn]])
        ax.imshow(mat, cmap="Blues", vmin=0, vmax=max(1, mat.max()))
        for (i, j), v in np.ndenumerate(mat):
            den = r.n_positive if i == 0 else r.n_negative
            ax.text(j, i, f"{v}\n({v}/{den})", ha="center", va="center", fontsize=14, color="white" if v > mat.max() / 2 else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["mould found", "no mould"]); ax.set_yticks([0, 1])
        ax.set_yticklabels([f"mould folder\n(n={r.n_positive})", f"clean folder\n(n={r.n_negative})"])
        ax.set_title(f"{k}\n({r.model_kind})")
    fig.suptitle(f"Confusion matrices at confidence >= {PRIMARY:.2f} (rows: provisional folder label)", fontsize=14)
    fig.text(0.5, 0.02, f"Labels are {LABEL_NOTE}.", ha="center", fontsize=10)
    fig.tight_layout(rect=(0, 0.05, 1, 0.93)); fig.savefig(FIG / "confusion_matrices.png", dpi=150); plt.show()

## Step 7. Same photos, every model
Original photo in the first column, then each model. Two clean-folder photos, two non-cobweb mould-folder photos and the two **hard** ones (the photos most models get wrong at 0.50; if none, the ones closest to the threshold). Boxes at or above 0.25: red = mapped mould, green = mapped healthy, grey = not mapped. Classifiers have no boxes: their top classes are written under the photo.

In [ ]:
# 10. Qualitative grid
if not HAVE_RESULTS:
    print("No results yet: grid not built.")
else:
    import matplotlib.patches as mp
    ok_img = IMG[(IMG.status == "ok") & IMG.image_id.isin(COMMON) & IMG.model_key.isin(usable)]
    wrong = ok_img.assign(w=ok_img[f"outcome_{PRIMARY:.2f}"].isin(["FP", "FN"])).groupby("image_id").w.sum()
    near = ok_img.assign(d=(ok_img.max_mould_conf - PRIMARY).abs()).groupby("image_id").d.mean()
    lab = SCORED.set_index("image_id")
    sha = lab.sha256.to_dict()
    order = sorted(COMMON, key=lambda i: (-wrong[i], near[i], sha[i]))
    hard = [i for i in order if wrong[i] > 0][:2] or sorted(COMMON, key=lambda i: (near[i], sha[i]))[:2]
    pick_c = [i for i in sorted(COMMON, key=lambda i: sha[i]) if lab.label[i] == "healthy" and i not in hard][:2]
    pick_m = [i for i in sorted(COMMON, key=lambda i: sha[i]) if lab.label[i] == "contaminated" and i not in hard][:2]
    picks = [("clean folder", i) for i in pick_c] + [("mould folder", i) for i in pick_m] + [("hard", i) for i in hard]
    if len(pick_c) < 2 or len(pick_m) < 2 or len(hard) < 2:
        print("note: fewer than 2 photos available in some group:", len(pick_c), len(pick_m), len(hard))
    paths = SCORED.set_index("image_id").local_path.to_dict()
    colour = {"suspected_contamination": "#d62728", "healthy": "#2ca02c", "ignore": "#888888", None: "#888888"}
    fig, axes = plt.subplots(len(picks), 1 + len(usable), figsize=(13.33, 1.65 * len(picks) + 0.9), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for r_i, (grp, iid) in enumerate(picks):
        im = load_rgb(paths[iid]); sc = 360 / max(im.size); th_im = im.resize((int(im.width * sc), int(im.height * sc)))
        axes[r_i][0].imshow(th_im); axes[r_i][0].set_title(f"{grp}: {iid[:34]}\nfolder label: {lab.label[iid]}", fontsize=8)
        for c_i, k in enumerate(usable, 1):
            ax = axes[r_i][c_i]; ax.imshow(th_im)
            c = by_model[k][lab.sha256[iid]]
            res = IMG[(IMG.model_key == k) & (IMG.image_id == iid)].iloc[0]
            for p in c["preds"]:
                if p["conf"] < min(THRESHOLDS):
                    continue
                if p["x"] is not None:
                    x0, y0 = (p["x"] - p["w"] / 2) * sc, (p["y"] - p["h"] / 2) * sc
                    ax.add_patch(mp.Rectangle((x0, y0), p["w"] * sc, p["h"] * sc, fill=False, lw=2, ec=colour[map_class(k, p["cls"])]))
                    ax.text(x0, max(y0, 8), f"{p['cls']} {p['conf']:.2f}", color="white", fontsize=6, bbox=dict(fc=colour[map_class(k, p["cls"])], ec="none", pad=0.8))
            said = res[f"pred_contaminated_{PRIMARY:.2f}"]; right = res[f"outcome_{PRIMARY:.2f}"] in ("TP", "TN")
            tops = "; ".join(f"{p['cls']} {p['conf']:.2f}" for p in sorted(c["preds"], key=lambda p: -p["conf"])[:2]) if c["kind"] == "classifier" else ""
            ax.set_title(f"{k}: {'MOULD' if said else 'no mould'} @{PRIMARY:.2f} ({'right' if right else 'WRONG'})" + (f"\n{tops}" if tops else ""),
                         fontsize=8, color="#1a7f37" if right else "#c0392b")
    fig.suptitle("Same photos for every model (red = mould class, green = healthy class, grey = not mapped)", fontsize=12)
    fig.text(0.5, 0.005, f"Right/wrong is against the {LABEL_NOTE}.", ha="center", fontsize=8)
    fig.tight_layout(rect=(0, 0.015, 1, 0.97)); fig.savefig(FIG / "qualitative_grid.png", dpi=150); plt.show()

## Step 8. Presentation summary
`presentation_summary.md` is written from the real numbers only. It states that the `cobweb/` folder was excluded. If inference has not been run it says "no results yet".

In [ ]:
# 11. presentation_summary.md
def words(t):
    return len(re.findall(r"\S+", t))

if not HAVE_RESULTS:
    text = ("# Pretrained mould models: no results yet\n\nInference has not been run, or no model answered with a class map set. "
            f"{len(MANIFEST)} test photos are listed in image_manifest.csv ({LABEL_NOTE}).\n")
else:
    prim = COMP[(COMP.subset == "all_common") & (COMP.threshold == PRIMARY)].set_index("model_key")
    loose = COMP[(COMP.subset == "all_common") & (COMP.threshold == min(THRESHOLDS))].set_index("model_key")
    nfl = int((SCORED.photo_flags != "").sum())
    unusable = [f"{r.model_key} ({r.status})" for r in REG.itertuples() if r.model_key not in usable]
    lines = [f"# Pretrained mould models on our test photos without cobweb\n",
             f"Inference only, nothing trained. The cobweb/ folder was excluded; only clean/ and mould/ photos were scored. {len(SCORED)} photos labelled by folder (provisional, not lab-confirmed); "
             f"{len(COMMON)} were answered by every usable model ({int(prim.n_positive.iloc[0])} mould-folder, {int(prim.n_negative.iloc[0])} clean-folder). "
             f"Models used: {', '.join(usable)}." + (f" Not in the results: {'; '.join(unusable)}." if unusable else ""),
             f"\nAt confidence {PRIMARY:.2f} (image counts as mould if one mapped mould box passes):\n"]
    for k in usable:
        r = prim.loc[k]
        lines.append(f"- {k} ({r.model_kind}): mould found {pct(r.sensitivity, r.sensitivity_frac)}, clean kept clear "
                     f"{pct(r.specificity, r.specificity_frac)}, balanced accuracy {pct(r.balanced_accuracy, r.balanced_accuracy_frac)}.")
    lines.append(f"\nAt {min(THRESHOLDS):.2f}: " + "; ".join(
        f"{k} mould found {loose.loc[k].sensitivity_frac}, clean kept clear {loose.loc[k].specificity_frac}" for k in usable) + ".")
    lines.append(f"\nCaution: the set is small, so differences of a photo or two are noise and we do not rank the models. {nfl} photo(s) are flagged as possible "
                 "training overlap or our own batch; model_comparison.csv also gives the numbers without them. Class maps were set by us "
                 "(see model_registry.csv); unmapped classes never count as mould. A classifier is not like-for-like with a detector.")
    text = "\n".join(lines) + "\n"
    n = words(text)
    print("summary words:", n, "(target 150-250)")
    if not 150 <= n <= 250:
        print("WARNING: summary length is outside 150-250 words; edit the template in this cell.")
(OUTPUT_DIR / "presentation_summary.md").write_text(text)
print(text)
print("files in", OUTPUT_DIR, ":", sorted(p.name for p in OUTPUT_DIR.iterdir()))

## Download
Kaggle: in the editor, open the **Output** panel on the right (`/kaggle/working`) and download the folder `mould_benchmark`. (If you use *Save Version* instead, set `CONFIRM_RUN = True` first, because a saved run starts with an empty cache.) It holds the CSVs, `excluded_photos.csv` (the cobweb images skipped), `figures/`, `presentation_summary.md` and `cache/` (raw replies, no key). Keep `cache/` so a later run, or the swap of the clean photos, reuses the calls.